# USB CT: sparse-view 3D reconstruction

Run CIL FDK, SIRT (1000 iterations), and FaCT-GS (20000 steps) independently using 25, 50, or 75 training views and 20 shared test views. This notebook contains the execution steps; historical results and comparison figures are in [EXPERIMENT_RECORD.md](EXPERIMENT_RECORD.md).

**CIL version:** `26.0.1.dev11+gb39f95799`. **Dataset:** `gruppe 4_tomo-A.txrm` (not bundled; public source and reuse terms pending). **Author/license:** local draft, pending confirmation.

Execution flags default to off. Only the optional single-view preprocessing preview displays images inline. Reconstruction volumes, projection images, and loss curves are saved to disk; the final table reports only newly evaluated results.

## 1. Configure paths and execution

Select a kernel containing CIL/TIGRE, NumPy and Matplotlib for direct reconstruction. See README.md for registering the existing `fact-gs` environment.

Start with 25 views and enable one method at a time. New results go to `runs_step_by_step/`. Rerun setup and geometry after changing `VIEWS`. Existing output directories are not overwritten.

In [ ]:
from pathlib import Path
import os, sys, json, time, subprocess
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

CASE = Path.cwd().resolve()
assert (CASE / 'assets/export_manifest.json').is_file(), 'Open Jupyter in the case directory.'
SCRIPTS = CASE / 'script/usb_dataset_gaussian'
sys.path.insert(0, str(SCRIPTS))
SPLIT_FILE = CASE / 'assets/export_manifest.json'
SOURCE = Path(os.environ.get('USB_CIL_DATA', CASE / 'dataset/usb_cil')).resolve()
RAW_TXRM = Path(os.environ.get('USB_TXRM', CASE / 'dataset/gruppe 4_tomo-A.txrm')).resolve()
RUN_ROOT = CASE / 'runs_step_by_step'
FACTGS_REPO = CASE / 'third_party/fact-gs'
RECON_PYTHON = sys.executable  # Use the selected kernel's environment.
VIEWS = 25
GPU = '1'  # Select an available GPU before importing CIL/TIGRE; restart the kernel to change it.
RUN_RAW_PREVIEW = False
RUN_FDK = False
RUN_SIRT = False
RUN_FACTGS = False
RUN_METRICS = False
assert VIEWS in (25, 50, 75)
os.environ['CUDA_VISIBLE_DEVICES'] = GPU
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
OUT = RUN_ROOT / f'{VIEWS}views'
split = json.loads(SPLIT_FILE.read_text())

# Clear previous reconstruction state if configuration is rerun.
data = ig = None

def run_script(name, *args):
    env = dict(os.environ)
    subprocess.run([RECON_PYTHON, str(SCRIPTS / name), *map(str, args)],
                   cwd=CASE, env=env, check=True)

print('Notebook kernel:', sys.executable)
print('Input:', SOURCE)
print('New outputs:', OUT)
print('Views:', VIEWS, '| GPU:', GPU)

## 2. Read a raw projection and apply CIL preprocessing

`ZEISSDataReader` already divides by the reference and applies recorded integer pixel shifts. Apply the negative logarithm **once**, then crop 200 pixels from each horizontal side. Preserve negative attenuation values. The 4×4 block average is applied **after** the logarithm.

Enable `RUN_RAW_PREVIEW` to read one view and see each step. This is a preview, not full dataset preparation. For a new scan export, the supporting `prepare.py` performs these same operations in batches (README.md gives the command). The reconstruction cells reuse the already prepared full scan in `SOURCE`.

In [ ]:
if RUN_RAW_PREVIEW:
    from cil.io import ZEISSDataReader
    from cil.processors import TransmissionAbsorptionConverter, Slicer
    assert RAW_TXRM.is_file(), RAW_TXRM
    transmission = ZEISSDataReader(file_name=str(RAW_TXRM), roi={'angle': (0, 1)}).read()
    converter = TransmissionAbsorptionConverter(white_level=1.0, min_intensity=1e-6, accelerated=False)
    absorption = converter(transmission)
    cropped = Slicer(roi={'horizontal': (200, -200)})(absorption)
    image = np.asarray(cropped.as_array(), dtype=np.float32).reshape(1024, 624)
    binned = image.reshape(256, 4, 156, 4).mean(axis=(1, 3))
    assert np.isfinite(binned).all()
    print('Transmission:', transmission.shape, '| Cropped:', image.shape, '| Binned:', binned.shape)
    window = np.percentile(image, [1, 99])
    fig, axes = plt.subplots(1, 2, figsize=(9, 5))
    for ax, array, title in zip(axes, [image, binned], ['Negative log + crop', '4×4 block average']):
        ax.imshow(array, origin='lower', cmap='gray', vmin=window[0], vmax=window[1])
        ax.set_title(title); ax.axis('off')
    plt.show()

## 3. Inspect the prepared data and fixed split

Prepared projections have shape `[angle, vertical, horizontal] = [801, 1024, 624]`. Only selected views are loaded for reconstruction. The final near-duplicate 360° endpoint is excluded. The 20 test views are outside the union of all three training sets.

In [ ]:
train_indices = np.asarray(split['experiments'][str(VIEWS)]['train_source_indices'])
test_indices = np.asarray(split['test_source_indices'])
assert len(train_indices) == VIEWS and len(test_indices) == 20
assert not np.intersect1d(train_indices, test_indices).size
print('Training views:', len(train_indices), '| Test views:', len(test_indices))
print('Training indices:', train_indices)
print('Test indices:', test_indices)
if (SOURCE / 'projections.npy').is_file():
    projections = np.load(SOURCE / 'projections.npy', mmap_mode='r')
    angles = np.load(SOURCE / 'angles.npy')
    print('Prepared array:', projections.shape, projections.dtype)
    print('Angle range (degrees):', np.rad2deg(angles[[0, -1]]))
else:
    print('Prepared data not found. Saved results remain viewable; configure SOURCE before reconstruction.')

## 4. Construct the reconstruction geometry

`load_data` restores the CIL source/detector geometry, selects training views, applies 4×4 averaging, and scales the longest object dimension to 2 without changing projection values. The common volume grid is ZYX `128×78×78`; density is inverse normalized length.

The helper only builds geometry and loads the fixed view indices; the reconstruction algorithms remain in the cells below. Inspect `script/usb_dataset_gaussian/reconstruct_cil.py` for its full implementation. The resulting acquisition and image geometries are printed below for a new CIL run.

In [ ]:
if RUN_FDK or RUN_SIRT:
    from functools import partialmethod
    from cil.framework import DataContainer
    from cil.plugins.tigre import FBP, ProjectionOperator
    from cil.optimisation.algorithms import SIRT
    from cil.version import version as cil_version
    from reconstruct_cil import load_data
    from prepare import geometry_record
    from export_cil_projections import export_projections
    DataContainer.sapyb = partialmethod(DataContainer.sapyb, num_threads=4)
    data, ig, selected_rows, original_indices, geometry_scale = load_data(
        SOURCE, train_indices, binning=4, resolution=128)
    np.testing.assert_array_equal(original_indices, train_indices)
    print('CIL version:', cil_version)
    print(data.geometry)
    print(ig)
    print('Projection shape:', data.shape, '| Volume shape:', ig.shape)
else:
    print('CIL reconstruction disabled. Expected input:', (VIEWS, 256, 156), '| Volume ZYX:', (128, 78, 78))

### Common output writer

Both CIL methods save a float32 volume, geometry, exact indices, and runtime. Projection export is a separate cell after each reconstruction. No PSNR/SSIM is calculated here. The manifest retains the fields used by the original projection exporter.

In [ ]:
def save_cil_result(folder, method, result, seconds):
    volume = np.asarray(result.as_array(), dtype=np.float32)
    assert volume.shape == (128, 78, 78) and np.isfinite(volume).all()
    np.save(folder / f'{method}_volume.npy', volume)
    np.save(folder / 'selected_rows.npy', selected_rows)
    np.save(folder / 'view_indices.npy', original_indices)
    np.save(folder / 'angles.npy', data.geometry.angles)
    (folder / 'acquisition_geometry.json').write_text(json.dumps(geometry_record(data.geometry), indent=2))
    manifest = dict(source=str(SOURCE.resolve()), method=method, views=VIEWS,
                    start_angle_deg=-180, angle_span_deg=360, binning=4,
                    volume_shape=list(volume.shape), volume_axis_order='ZYX',
                    voxel_size_xyz=[ig.voxel_size_x, ig.voxel_size_y, ig.voxel_size_z],
                    geometry_scale=float(geometry_scale), density_units='inverse normalized distance',
                    cil_version=cil_version, backend='CIL/TIGRE', cpu_threads=4,
                    sirt_iterations=1000 if method == 'sirt' else None,
                    sirt_initial='zero' if method == 'sirt' else None,
                    sirt_nonnegative=method == 'sirt', loss_interval=10 if method == 'sirt' else None,
                    metrics={method: {'reconstruction_seconds': seconds}})
    (folder / 'manifest.json').write_text(json.dumps(manifest, indent=2))
    print(f'{method.upper()}: {seconds:.3f} s; volume saved to {folder}')
    return volume

## 5. FDK — run independently

CIL/TIGRE `FBP` performs cone-beam FDK for this 3D geometry. Timing below covers only reconstruction; saving and forward projection are excluded.

In [ ]:
if RUN_FDK:
    assert data is not None, 'Run the geometry cell first.'
    folder = OUT / 'fdk'
    folder.mkdir(parents=True, exist_ok=False)
    start = time.perf_counter()
    fdk = FBP(ig, data.geometry)(data)
    elapsed = time.perf_counter() - start
    fdk_volume = save_cil_result(folder, 'fdk', fdk, elapsed)
else:
    print('FDK is disabled. Enable RUN_FDK and rerun setup/geometry to reconstruct.')

### Forward-project FDK and save every image

The exporter uses `ProjectionOperator` on the saved volume, for both training and held-out angles. Each measured/rendered view is saved as float32 NPY and display PNG, with original indices and angles. PNG windows do not affect metrics.

In [ ]:
if RUN_FDK:
    export_projections(OUT / 'fdk', SPLIT_FILE)
    print('FDK measured/rendered training and test projections exported.')

## 6. SIRT — run independently

Start from zero, enforce nonnegativity, and run **1000 iterations**. Recompute `0.5 * sum((Ax-b)**2)` from the current solution every ten iterations. This avoids mixing the initial residual with CIL's cached weighted residual in the loss curve.

In [ ]:
if RUN_SIRT:
    assert data is not None, 'Run the geometry cell first.'
    class ResidualSIRT(SIRT):
        def update_objective(self):
            residual = self.operator.direct(self.solution).as_array().astype(np.float64)
            residual -= self.data.as_array()
            self.loss.append(float(0.5 * np.sum(residual**2)))
            if not hasattr(self, 'loss_iterations'):
                self.loss_iterations = []
            self.loss_iterations.append(max(0, self.iteration))

    folder = OUT / 'sirt'
    folder.mkdir(parents=True, exist_ok=False)
    start = time.perf_counter()
    operator = ProjectionOperator(ig, data.geometry)
    solver = ResidualSIRT(initial=ig.allocate(0), operator=operator, data=data,
                          lower=0, update_objective_interval=10)
    def progress(algorithm):
        if algorithm.iteration % 100 == 0:
            print(f'SIRT iteration {algorithm.iteration}/1000', flush=True)
    solver.run(1000, callbacks=[progress], verbose=0)
    if solver.loss_iterations[-1] != solver.iteration:
        solver.update_objective()
    history = dict(iterations=solver.loss_iterations, loss=solver.loss,
                   projection_pixels=int(data.as_array().size),
                   definition='0.5 * sum((A x - b)^2), recomputed at current solution')
    (folder / 'sirt_loss.json').write_text(json.dumps(history, indent=2))
    elapsed = time.perf_counter() - start
    sirt_volume = save_cil_result(folder, 'sirt', solver.solution, elapsed)
else:
    print('SIRT is disabled.')

In [ ]:
if RUN_SIRT:
    export_projections(OUT / 'sirt', SPLIT_FILE)
    steps = np.asarray(history['iterations'])
    mse = 2 * np.asarray(history['loss']) / history['projection_pixels']
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.semilogy(steps, mse)
    ax.set(xlabel='Completed iterations', ylabel='Projection residual MSE')
    ax.grid(alpha=0.3)
    fig.savefig(OUT / 'sirt/sirt_loss.png', bbox_inches='tight')
    plt.close(fig)
    print('Loss curve saved:', OUT / 'sirt/sirt_loss.png')

## 7. FaCT-GS — convert and train independently

The converter uses the **same fixed indices, block averaging, normalized coordinates, and volume grid**. It flips rows to the Gaussian camera convention; export flips them back to CIL. The saved voxel volume is transposed to ZYX. No 3D ground truth is fabricated.

The bundled optimization loop in `third_party/fact-gs` is invoked by `train.py` in a separate process: 20000 steps, 10000 initial Gaussians, seed 0, FDK gradient initialization, and `L1 + 0.3 × (1-SSIM) + 0.05 × TV`. The USB reader and result writer are passed explicitly to the bundled trainer; no global callback table is replaced. Conversion and training are separate cells, so training can be retried without converting again.

In [ ]:
converted = RUN_ROOT / f'datasets/factgs_{VIEWS}views'
if RUN_FACTGS:
    assert (FACTGS_REPO / 'train_recon.py').is_file(), 'Bundled third_party/fact-gs source is missing.'
    run_script('convert.py', '--source', SOURCE, '--output', converted,
               '--train-views', VIEWS, '--test-views', 20, '--binning', 4,
               '--resolution', 128, '--split-manifest', SPLIT_FILE)
    print('FaCT-GS input saved:', converted)

In [ ]:
if RUN_FACTGS:
    run_script('train.py', '--data', converted, '--output', OUT / 'factgs',
               '--steps', 20000, '--gaussians', 10000)
    assert (OUT / 'factgs/COMPLETE').is_file()
    print('All measured and Gaussian-rendered projections saved as NPY/PNG.')

## 8. Compute projection metrics separately

Only evaluate **saved float32 arrays**, never PNG images. `evaluate_projections.py` validates shapes, indices, angles, and disjoint splits. Use the same `data_range = 3.142874300` across all methods. PSNR and SSIM are computed per view then averaged; SSIM uses a 7×7 uniform window and sample covariance.

Select the completed methods below. This step is independent of reconstruction flags and evaluates the current `VIEWS`. Repeat for 25/50/75 views. These are projection-consistency metrics against noisy measurements, not 3D ground-truth scores.

In [ ]:
METHODS_TO_EVALUATE = ['fdk', 'sirt', 'factgs']  # Restrict to finished methods if needed.
if RUN_METRICS:
    for method in METHODS_TO_EVALUATE:
        result = OUT / method
        assert (result / 'projections/test_measured/projections.npy').is_file(), f'Missing projections: {result}'
        run_script('evaluate_projections.py', '--data-root', result / 'projections',
                   '--methods', method, '--data-range', 3.142874300,
                   '--output', result / 'projection_metrics.json')
    print('New metrics saved under', OUT)
else:
    print('Enable RUN_METRICS to evaluate saved outputs from this run.')

In [ ]:
if RUN_METRICS:
    rows = ['| Method | Train PSNR | Train SSIM | Test PSNR | Test SSIM |',
            '| --- | ---: | ---: | ---: | ---: |']
    for method in METHODS_TO_EVALUATE:
        record = json.loads((OUT / method / 'projection_metrics.json').read_text())
        q = record['results'][str(VIEWS)][method]
        a, b = q['train'], q['test']
        rows.append(f"| {method} | {a['psnr_mean_db']:.4f} | {a['ssim_mean']:.4f} | {b['psnr_mean_db']:.4f} | {b['ssim_mean']:.4f} |")
    display(Markdown('\n'.join(rows)))